In [ ]:
import openai
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import math
import re
import time

openai.api_key = ''
# --- Load Data ---
data = pd.read_csv("data_analysis/data/creativity_data/cleaned_data_with_comments.csv")['text'].dropna().tolist()

# --- Extraction helpers ---
_score_re = re.compile(r"creativityperception\s*:\s*(\d+)", re.IGNORECASE)
_reason_re = re.compile(r"reason\s*:\s*(.*)", re.IGNORECASE)

def parse_output(raw, comment):
    if not isinstance(raw, str) or not raw.strip():
        return {
            "comment": comment,
            "creativity_score": None,
            "reason": "No content from GPT",
            "raw_output": raw
        }

    # Tolerant regex
    score_match = re.search(r"creativity\s*perception\s*[:\-]?\s*(\d+)", raw, re.IGNORECASE)
    reason_match = re.search(r"reason\s*[:\-]?\s*(.*)", raw, re.IGNORECASE)

    score = int(score_match.group(1)) if score_match else None
    reason = reason_match.group(1).strip() if reason_match else None

    return {
        "comment": comment,
        "creativity_score": score,
        "reason": reason,
        "raw_output": raw
    }

# --- GPT creativity scorer ---
def rate_creativity(comment):
    raw = None
    try:
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            temperature=0,
            max_tokens=100,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are evaluating the strength of admiration to creativity a comment expresses toward the specific Wordle gameplay being discussed (either the original poster's or the commenter's own play).\n"
                        "Rate on a scale from 1 to 10 based on admiration to a creative play:\n"
                        "- High admiration (7-10): Strong, clear praise or excitement about the gameplay, even if short (e.g., 'Amazing', 'Nice', 'That's smart').\n"
                        "- Low admiration (1-3): No praise, unrelated comment, neutral statement, criticism, or advice.\n"
                        "- Middle admiration (4-6): Mild or ambiguous engagement — some strategies involved but lack of strong applause.\n"
                        "Focus on tone and intent toward the specific game shown, not general gameplay habits or unrelated strategy tips."
                    )

                },
                {
                    "role": "user",
                    "content": (
                        "Rate from 1 (no creativity/strategy) to 10 (strong perception of creativity/strategy):\n\n"
                        f"Comment: \"{comment}\"\n\n"
                        "Respond in format:\n"
                        "CreativityPerception: <1-10>\n"
                        "Reason: <one concise sentence>"
                    )
                }
            ]
        )

        raw = response.choices[0].message['content'].strip()
        return parse_output(raw, comment)

    except Exception as e:
        # Normalize the error string to help detect rate limits
        err_str = str(e)
        return {
            "comment": comment,
            "creativity_score": None,
            "reason": f"Error: {err_str}",
            "raw_output": raw
        }

# --- Batch processing with rate-limit stop ---
batch_size = 300
num_batches = math.ceil(len(data) / batch_size)
max_workers = 3
os.makedirs("data_analysis/data/creativity_data", exist_ok=True)

overall_start = time.time()
stopped_due_to_rate = False

def is_rate_limit_reason(reason: str) -> bool:
    if not isinstance(reason, str):
        return False
    r = reason.lower()
    return ("rate limit" in r) or ("rate_limit" in r) or ("too many requests" in r)

processed_overall = 0  # count of results written (across batches)

for batch_idx in range(num_batches):
    start_idx = batch_idx * batch_size
    end_idx = min((batch_idx + 1) * batch_size, len(data))
    batch_data = data[start_idx:end_idx]

    print(f"\nProcessing batch {batch_idx+1}/{num_batches} "
          f"({len(batch_data)} comments: index {start_idx} to {end_idx-1})")

    results = []
    progress_bar = tqdm(total=len(batch_data), desc=f"Batch {batch_idx+1}")

    # Rate limit tracking
    consecutive_rate_errors = 0
    batch_start = time.time()

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(rate_creativity, c): c for c in batch_data}

        for future in as_completed(futures):
            result = future.result()
            results.append(result)
            progress_bar.update(1)

            # Check for rate limit in this result
            if result and isinstance(result, dict) and is_rate_limit_reason(result.get("reason", "")):
                consecutive_rate_errors += 1
            else:
                consecutive_rate_errors = 0

            # If >5 rate-limit errors in a row, save partial and stop
            if consecutive_rate_errors > 5:
                progress_bar.close()
                batch_elapsed = time.time() - batch_start

                # Save partial batch
                df_partial = pd.DataFrame(results)
                partial_name = f"data_analysis/data/creativity_data/gpt_batch{batch_idx+1}_creativityperception_rated_PARTIAL_STOP.csv"
                df_partial.to_csv(partial_name, index=False)

                processed_overall += len(results)

                print("\n===== EARLY STOP TRIGGERED: Rate limit errors =====")
                print(f"Consecutive rate-limit errors: {consecutive_rate_errors}")
                print(f"Saved partial batch to: {partial_name}")
                print(f"Processed {len(results)} / {len(batch_data)} in this batch "
                      f"({processed_overall} / {len(data)} overall).")
                print(f"Batch runtime: {batch_elapsed:.1f}s | Total runtime: {overall_elapsed:.1f}s")
                stopped_due_to_rate = True
                break 

    # if we stopped early inside this batch, stop the whole run
    if stopped_due_to_rate:
        break

    progress_bar.close()

    # Save this completed batch
    df = pd.DataFrame(results)
    if "comment" not in df.columns:
        df["comment"] = None
    if "raw_output" not in df.columns:
        df["raw_output"] = None

    batch_filename = f"data_analysis/data/creativity_data/gpt_batch{batch_idx+1}_creativityperception_rated.csv"
    df.to_csv(batch_filename, index=False)

    processed_overall += len(results)
    print(f"Saved {len(df)} comments to {batch_filename}")
    print(f"Debug raw outputs saved to {debug_filename}")

# Final summary
total_elapsed = time.time() - overall_start
if stopped_due_to_rate:
    print("\nRUN STOPPED DUE TO REPEATED RATE LIMITS.")
else:
    print("\nRUN COMPLETED.")

print(f"Processed {processed_overall} / {len(data)} comments total.")
print(f"Total runtime: {total_elapsed:.1f}s")
